# Curve fitting

Curve fitting means adjusting a model's parameters so that its predictions are close to the observed data. Neural networks use the same basic idea, but with more flexible models and many more parameters.

In this notebook, you will manually tune three simple models:

1. a line with one input feature;
2. a plane with two input features; and
3. a logistic regression model for binary classification.

> **Using this notebook in Google Colab:** select **Runtime → Run all**, then move the sliders. You do not need to edit the code. If a widget does not appear, run its cell again with the play button on the left.

The data are synthetic, so they are intended for learning rather than clinical interpretation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Layout, widgets, interact
from IPython.display import display

SLIDER_STYLE = {'description_width': 'initial'}
SLIDER_LAYOUT = Layout(width='550px')


def mean_squared_error(observed, predicted):
    """Average squared distance between observations and predictions."""
    return np.mean((observed - predicted) ** 2)


def calculate_bce_loss(y_true, y_pred):
    """
    Calculate the binary cross-entropy loss.

    Parameters:
    -----------
    y_true (array-like): True binary labels (0 or 1).
    y_pred (array-like): Predicted probabilities, between 0 and 1.

    Returns:
    --------
    float: The average binary cross-entropy loss.
    """
    # Ensure that y_pred does not contain values exactly equal to 0 or 1,
    # as log(0) is undefined and can cause computation errors.
    epsilon = 1e-10
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    # Calculate binary cross-entropy loss
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss


## Activity 1: Fit a line

The model is `prediction = slope × x + intercept`. Move the sliders to place the red line through the data. Your goal is to make the **mean squared error (MSE)** as small as possible.

In [ ]:
# Create reproducible data that roughly follow a straight line.
rng = np.random.default_rng(42)
line_x = rng.uniform(-10, 10, size=500)
line_y = 2 * line_x + 1 + rng.normal(0, 2, size=line_x.size)
line_x_for_plot = np.linspace(line_x.min(), line_x.max(), 200)


@interact(
    slope=FloatSlider(value=-2.0, min=-2.5, max=2.5, step=0.05, style=SLIDER_STYLE, layout=SLIDER_LAYOUT, description='Slope:', continuous_update=False),
    intercept=FloatSlider(value=-10, min=-10, max=10, step=0.1, style=SLIDER_STYLE, layout=SLIDER_LAYOUT, description='Intercept:', continuous_update=False),
)
def plot_line_fit(slope, intercept):
    predicted_y = slope * line_x + intercept
    fitted_line_y = slope * line_x_for_plot + intercept
    mse = mean_squared_error(line_y, predicted_y)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(line_x, line_y, s=18, alpha=0.35, label='Observed data')
    ax.plot(line_x_for_plot, fitted_line_y, color='crimson', linewidth=3, label='Model')
    ax.set(
        title=f'Fit a line — MSE: {mse:.2f} (lower is better)',
        xlabel='Input feature (x)',
        ylabel='Outcome (y)',
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()
    plt.close(fig)


## Activity 2: Fit a plane using two input features

The model is `prediction = a × feature 1 + b × feature 2 + bias`. Adjust **a**, **b**, and **bias** until the surface fits the points. The orientation sliders only change your view; they do not change the model.

In [ ]:
# Create a plotting grid and reproducible points from a plane.
plane_axis = np.linspace(-10, 10, 20)
plane_grid_x, plane_grid_y = np.meshgrid(plane_axis, plane_axis)

rng = np.random.default_rng(42)
plane_x = rng.uniform(-10, 10, size=100)
plane_y = rng.uniform(-10, 10, size=100)
plane_z = plane_x + plane_y


@interact(
    a=FloatSlider(value=0, min=-2, max=2, step=0.1, description='Coefficient a:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
    b=FloatSlider(value=0, min=-2, max=2, step=0.1, description='Coefficient b:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
    bias=FloatSlider(value=0, min=-5, max=5, step=0.1, description='Bias:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
    azimuth=FloatSlider(value=45, min=0, max=360, step=5, description='Rotate:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
    elevation=FloatSlider(value=30, min=0, max=90, step=5, description='Tilt:', style=SLIDER_STYLE, layout=SLIDER_LAYOUT, continuous_update=False),
)
def plot_plane_fit(a, b, bias, azimuth, elevation):
    surface_z = a * plane_grid_x + b * plane_grid_y + bias
    predicted_z = a * plane_x + b * plane_y + bias
    mse = mean_squared_error(plane_z, predicted_z)

    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(projection='3d')
    ax.plot_surface(
        plane_grid_x, plane_grid_y, surface_z,
        cmap='viridis', edgecolor='none', alpha=0.55,
    )
    ax.scatter(plane_x, plane_y, plane_z, color='crimson', s=22, label='Observed data')
    ax.set(
        title=f'Fit a plane — MSE: {mse:.2f} (lower is better)',
        xlabel='Feature 1',
        ylabel='Feature 2',
        zlabel='Outcome',
    )
    ax.view_init(elev=elevation, azim=azimuth)
    ax.legend()
    plt.show()
    plt.close(fig)


## Class Activity: Logistic Regression

In [ ]:
# Define the logistic regression function
def logistic_regression(feature_1, feature_2, weights, bias):
    logit = (weights[0] * feature_1) + (weights[1] * feature_2) + bias
    return 1 / (1 + np.exp(-logit)) # sigmoid function

In [ ]:
# Generate synthetic data
np.random.seed(42)
# Class 0
feature_0_class_0 = np.random.normal(2, 1, 100)  # Feature 1 for class 0
feature_1_class_0 = np.random.normal(2, 1, 100)  # Feature 2 for class 0
# Class 1
feature_0_class_1 = np.random.normal(5, 1, 100)  # Feature 1 for class 1
feature_1_class_1 = np.random.normal(5, 1, 100)  # Feature 2 for class 1

features = np.vstack((np.column_stack((feature_0_class_0, feature_1_class_0)),
                      np.column_stack((feature_0_class_1, feature_1_class_1))))
y_true = np.array([0]*100 + [1]*100)

In [ ]:
# Grid for decision boundary visualization
feature_0, feature_1 = np.meshgrid(
    np.linspace(
        min(np.concatenate([feature_0_class_0, feature_0_class_1])), 
        max(np.concatenate([feature_0_class_0, feature_0_class_1])), 
        50
    ),
    np.linspace(
        min(np.concatenate([feature_1_class_0, feature_1_class_1])), 
        max(np.concatenate([feature_1_class_0, feature_1_class_1])), 
        50
    ),
)

@interact(
    weight1=FloatSlider(value=1, min=-15, max=15, step=0.01, style=SLIDER_STYLE, layout=SLIDER_LAYOUT, description='Weight 1:', continuous_update=False),                 
    weight2=FloatSlider(value=-1, min=-15, max=15, step=0.01, style=SLIDER_STYLE, layout=SLIDER_LAYOUT, description='Weight 2:', continuous_update=False),       
    bias=FloatSlider(value=0, min=-20, max=20, step=0.01, style=SLIDER_STYLE, layout=SLIDER_LAYOUT, description='Bias:', continuous_update=False),  
)
def plot_decision_boundary(weight1, weight2, bias):
    # Calculate z values for the contour
    zz = logistic_regression(feature_0, feature_1, [weight1, weight2], bias)
    zz = zz.reshape(feature_0.shape)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(
        features[:,0][:y_true.size // 2], 
        features[:,1][:y_true.size // 2], 
        c='blue',
        label='Class: 0',
        alpha=0.8
    )
    plt.scatter(
        features[:,0][y_true.size // 2:], 
        features[:,1][y_true.size // 2:], 
        c='red',
        label='Class: 1',
        alpha=0.8
    )
    
    contour = plt.contourf(
        feature_0, 
        feature_1,    
        zz, 
        levels=[0, 0.5, 1],
        cmap="coolwarm", 
        alpha=0.3)
    
    plt.colorbar(contour)
    # Decision boundary line for the zz = 0.5 threshold
    plt.contour(
        feature_0, 
        feature_1,
        zz, 
        levels=[0.5], 
        colors='k', 
        vmin=0, vmax=1, 
        linestyles='dashed')
    
    plt.title('Interactive Logistic Regression Decision Boundary')
    plt.xlabel('Feature 0')
    plt.ylabel('Feature 1')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
weight_1 = widgets.FloatText(
    value=0.0,
    description='Weight 1:',
    step=0.1,
    style={'description_width': 'initial'}
)

weight_2 = widgets.FloatText(
    value=0.0,
    description='Weight 2:',
    step=0.1,
    style={'description_width': 'initial'}
)

bias = widgets.FloatText(
    value=0.0,
    description='Bias',
    step=0.1,
    style={'description_width': 'initial'}
)

# Display the widget
display(weight_1, weight_2, bias)

# Output widget to display the results
output = widgets.Output()
display(output)

# Function to update the output based on the inputs
def update_output(*args):
    with output:
        output.clear_output()
        
        y_pred = logistic_regression(
            features[:, 0], 
            features[:, 1], 
            weights=[weight_1.value, weight_2.value], 
            bias=bias.value
        )
            
        # Calculate loss
        loss = calculate_bce_loss(y_true, y_pred)
        
        print(f"Weight 1 Value: {weight_1.value}")
        print(f"Weight 2 Value: {weight_2.value}")
        print(f"Bias Value: {bias.value}")

        print(f"BCE LOSS (lower better): {loss:.4f}")

# Attach the observer to the 'value' trait of the float_input widget
# Observe changes in each widget and call update_output when any change happens
weight_1.observe(update_output, names='value')
weight_2.observe(update_output, names='value')
bias.observe(update_output, names='value')

## End.